# SoftMax回归 - 实际是分类问题

分类问题中，针对于分多少类，需要输出多少个值
每个值代表一个类别的置信度


Softmax 函数是深度学习中处理多分类问题的核心组件，它通过数学转换将神经网络的原始输出（Logits）转化为概率分布。

$$\hat{y}_j = \text{softmax}(z_j) = \frac{e^{z_j}}{\sum_{k=1}^{K} e^{z_k}}$$

为什么要用指数函数表示，因为指数函数非负，然后通过除以所有的原值的指数值来得到概率，得到全部类别加起来为1的特性。指数函数还具有“拉大差距”的效应。得分稍高的类别在经过指数运算后，其对应的概率会显著高于得分较低的类别，这有助于模型更明确地进行分类判断。

In [1]:
# 获取数据集
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

# 通过ToTensor实例将图像数据从PIL类型变换成32位浮点数格式，
# 并除以255使得所有像素的数值均在0～1之间
trans = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(
    root="../data", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, transform=trans, download=True)

# len(mnist_train), len(mnist_test)
print(len(mnist_train), len(mnist_test))


def get_fashion_mnist_labels(labels):  #@save
    """返回Fashion-MNIST数据集的文本标签"""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
                   'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]


batch_size = 256

def get_dataloader_workers():  #@save
    """使用4个进程来读取数据"""
    return 4

train_iter = data.DataLoader(mnist_train, batch_size, shuffle=True,
                             num_workers=get_dataloader_workers())
# data.DataLoader函数是PyTorch中用于加载数据的工具，它可以将数据集分成小批量，并在训练过程中进行迭代。参数说明如下：
# - mnist_train: 这是要加载的数据集，通常是一个PyTorch Dataset对象。
# - batch_size: 这是每个批次的数据量，即每次迭代时加载多少样本。
# - shuffle=True: 这表示在每个epoch开始时，数据将被打乱，以增加训练的随机性和泛化能力。
# - num_workers=get_dataloader_workers(): 这指定了用于加载数据的子进程数量。这可以加快数据加载的速度，特别是在处理大型数据集时。

60000 10000


In [2]:
def load_data_fashion_mnist(batch_size, resize=None):  #@save
    """下载Fashion-MNIST数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize)) # 如果指定了resize参数，则在转换列表的开头插入一个Resize转换，以调整图像的大小。
        # resize参数通常是一个整数或一个元组，表示调整图像的目标大小。例如，如果resize=64，则图像将被调整为64x64像素。
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True,
                            num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_test, batch_size, shuffle=False,
                            num_workers=get_dataloader_workers()))

In [3]:
import torch
from IPython import display


batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size) # 获取训练和测试数据迭代器


# 初始化模型参数
num_inputs = 784 # 因为每张图像是28x28像素，所以输入层的神经元数量是784
num_outputs = 10 # 因为Fashion-MNIST数据集有10个类别，所以输出层的神经元数量是10

# W代表权重矩阵，b代表偏置向量。权重矩阵W的形状是(num_inputs, num_outputs)，
# 即(784, 10)，表示每个输入特征与每个输出类别之间的连接权重。
# 偏置向量b的形状是(num_outputs,)，即(10,)，表示每个输出类别的偏置项。
# 为什么b为全零？在神经网络中，权重通常被初始化为小的随机值，以打破对称性并促进学习。而偏置项通常被初始化为零，因为它们不需要打破对称性，并且在训练过程中会根据数据自动调整。初始化为零的偏置项不会影响模型的学习能力，因为它们会在训练过程中根据数据进行更新。
# torch.normal函数用于生成一个张量，其中的元素是从指定的正态分布中随机抽取的。参数说明如下：
# - 0: 这是正态分布的均值（mean），
# - 0.01: 这是正态分布的标准差（standard deviation）
# - size: 这是生成的张量的形状
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
# torch.zeros函数用于生成一个全零的张量。参数说明如下：
# - num_outputs: 这是生成的张量的形状，这里是一个一维张量，长度为num_outputs
# - requires_grad=True: 这表示在计算图中需要跟踪这个张量的梯度，以便在反向传播过程中进行优化。
b = torch.zeros(num_outputs, requires_grad=True)

#  定义softmax操作
# 给定一个矩阵x，我们可以对所有元素求和来得到一个标量。
# 对于softmax函数，我们需要对每一行进行求和，以便对每一行的元素进行归一化。
# 通过设置keepdim=True，我们可以保持求和后的维度不变，这样我们就可以直接进行广播操作来计算softmax函数的输出。
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(X.sum(0, keepdim=True)) # 沿着列的方向求和，结果是一个1行3列的张量
print(X.sum(1, keepdim=True)) # 沿着行的方向求和，结果是一个2行1列的张量



tensor([[5., 7., 9.]])
tensor([[ 6.],
        [15.]])


## 实现softmax由三个步骤组成：

1. 对每个项求幂（使用exp）；

2. 对每一行求和（小批量中每个样本是一行），得到每个样本的规范化常数；

3. 将每一行除以其规范化常数，确保结果的和为1。

In [4]:
#  定义softmax操作
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition  # 这里应用了广播机制
    # partition会扩展成与X_exp相同的形状，以便进行逐元素除法，从而得到softmax函数的输出。

X = torch.normal(0, 1, (2, 5)) # 生成一个2行5列的张量，其中的元素是从均值为0、标准差为1的正态分布中随机抽取的。
print(X)
X_prob = softmax(X) 
# 通过观察输出，我们可以看到每一行的元素之和都等于1，这验证了softmax函数的正确性。
print(X_prob, X_prob.sum(1))

tensor([[-0.9828, -2.7440, -0.6460,  0.4024,  0.8034],
        [-0.3429, -0.6772,  0.1636,  2.0925,  0.5666]])
tensor([[0.0798, 0.0137, 0.1117, 0.3188, 0.4760],
        [0.0579, 0.0414, 0.0960, 0.6610, 0.1437]]) tensor([1., 1.])


In [9]:
# 定义模型
# 对输入X进行线性变换，即将输入X与权重矩阵W相乘，并加上偏置b。然后将结果传递给softmax函数，以得到每个类别的预测概率。
def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)
# reshape((-1, W.shape[0]))的作用是将输入X调整为一个二维张量，其中-1表示自动推断行数，W.shape[0]表示列数，即输入特征的数量（784）。这样做是为了确保输入数据的形状与权重矩阵W的形状兼容，以便进行矩阵乘法。

# 创建一个数据y_hat，其中包含2个样本在3个类别的预测概率，使用y作为y_hat中概率的索引
y = torch.tensor([0, 2]) # 示每个样本的真实类别标签。第一个样本的真实类别是0，第二个样本的真实类别是2。
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]]) # 这里的y_hat是一个2行3列的张量，表示每个样本在三个类别上的预测概率。第一行对应第一个样本的预测概率，第二行对应第二个样本的预测概率。
print(y_hat[[0, 1], y]) # 对于第一个样本（索引0），我们选择类别0的预测概率（0.1）；对于第二个样本（索引1），我们选择类别2的预测概率（0.5）。
# 因此，y_hat[[0, 1], y]的结果是一个包含两个元素的张量，分别是0.1和0.5。

# 用法总结
#tensor[[a, b, c, ...], [d, e, f, ...]] 的用法是从tensor中选择特定位置的元素。
# 第一个列表[a, b, c, ...]表示要选择的行索引，
# 第二个列表[d, e, f, ...]表示要选择的列索引。
# 这个操作会返回一个新的张量，其中包含了在指定行和列位置上的元素。 

# 定义损失函数, 实现交叉熵损失函数
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])
# torch.log函数用于计算输入张量的自然对数。在交叉熵损失函数中，我们需要计算预测概率的对数，以便在计算损失时使用。
# 该函数会返回一个新的张量，其中每个元素都是输入张量对应元素的自然对数值。
# 为什么没有乘p(x)，因为在交叉熵损失函数中，我们通常使用的是one-hot编码的标签，即每个样本的真实类别对应的位置是1，其他位置是0。因此，在计算损失时，我们只需要关注预测概率中对应真实类别的位置，而不需要乘以p(x)。

print(cross_entropy(y_hat, y))

def accuracy(y_hat, y):  #@save
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1: # 如果y_hat是一个二维张量，并且第二维的大小大于1，说明y_hat包含了每个样本在多个类别上的预测概率。
        y_hat = y_hat.argmax(axis=1) # 在这种情况下，我们使用argmax函数沿着第二维（类别维度）找到每个样本预测概率最大的类别索引，这样y_hat就变成了一个一维张量，包含了每个样本的预测类别索引。
        # 对于矩阵，axis=1表示沿着行的方向进行操作，即在每一行中找到最大值的索引。
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum()) # 这里的cmp是一个布尔张量，表示每个样本的预测类别是否与真实类别相匹配。通过将cmp转换为与y相同的数据类型（通常是整数类型），我们可以得到一个包含0和1的张量，其中1表示预测正确，0表示预测错误。最后，通过对这个张量进行求和，我们可以得到预测正确的样本数量，并将其转换为浮点数返回。
    # tensor.type(x.dtype)函数用于将张量转换为与另一个张量x相同的数据类型。在accuracy函数中，我们使用y_hat.type(y.dtype)将y_hat转换为与y相同的数据类型，以便进行比较操作。这样可以确保在比较预测类别索引和真实类别标签时，它们具有相同的数据类型，从而避免类型不匹配的问题。

print(accuracy(y_hat, y))
print(accuracy(y_hat, y) / len(y))


def evaluate_accuracy(net, data_iter):  #@save
    """计算在指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval()  # 将模型设置为评估模式
    metric = Accumulator(2)  # 正确预测数、预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel()) # numel()函数用于计算张量中元素的总数。在evaluate_accuracy函数中，我们使用y.numel()来获取当前批次中样本的数量，并将其添加到metric中，以便在计算精度时使用。
    return metric[0] / metric[1]

class Accumulator:  #@save
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n # data是一个长度为n的列表，用于存储累加的结果。每个元素初始化为0.0，表示初始状态下的累加值。

    def add(self, *args):
        for a, b in zip(self.data, args):
            print(a)
            print(b)
        # 通过zip函数将self.data和args中的元素一一对应地组合成一个个元组，然后使用列表推导式对每个元组中的元素进行相加，并将结果存储回self.data中。
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx): # __getitem__ 是一个特殊方法，用于定义对象的索引访问行为。当我们使用对象[索引]的方式访问对象时，Python会自动调用这个方法，并将索引作为参数传递给它。在
        return self.data[idx]

evaluate_accuracy(net, test_iter)



tensor([0.1000, 0.5000])
tensor([2.3026, 0.6931])
1.0
0.5
0.0
34.0
0.0
256
34.0
41.0
256.0
256
75.0
33.0
512.0
256
108.0
35.0
768.0
256
143.0
32.0
1024.0
256
175.0
35.0
1280.0
256
210.0
31.0
1536.0
256
241.0
33.0
1792.0
256
274.0
25.0
2048.0
256
299.0
29.0
2304.0
256
328.0
36.0
2560.0
256
364.0
48.0
2816.0
256
412.0
33.0
3072.0
256
445.0
30.0
3328.0
256
475.0
32.0
3584.0
256
507.0
38.0
3840.0
256
545.0
36.0
4096.0
256
581.0
41.0
4352.0
256
622.0
32.0
4608.0
256
654.0
25.0
4864.0
256
679.0
28.0
5120.0
256
707.0
35.0
5376.0
256
742.0
37.0
5632.0
256
779.0
38.0
5888.0
256
817.0
34.0
6144.0
256
851.0
41.0
6400.0
256
892.0
26.0
6656.0
256
918.0
41.0
6912.0
256
959.0
36.0
7168.0
256
995.0
32.0
7424.0
256
1027.0
28.0
7680.0
256
1055.0
37.0
7936.0
256
1092.0
34.0
8192.0
256
1126.0
29.0
8448.0
256
1155.0
32.0
8704.0
256
1187.0
43.0
8960.0
256
1230.0
38.0
9216.0
256
1268.0
29.0
9472.0
256
1297.0
33.0
9728.0
256
1330.0
7.0
9984.0
16


## 模型训练方法

In [ ]:
# net是模型，train_iter是训练数据的迭代器，loss是损失函数，updater是优化器。这个函数的作用是在一个迭代周期内训练模型，并返回训练损失和训练精度。
def train_epoch_ch3(net, train_iter, loss, updater):  #@save
    """训练模型一个迭代周期（定义见第3章）"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    # 训练损失总和、训练准确度总和、样本数
    metric = Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X) #    
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad() # 在使用PyTorch内置的优化器时，我们需要先调用zero_grad()方法将所有参数的梯度清零，以避免梯度累积的影响。
            l.mean().backward() # 
            updater.step() # 在调用step()方法时，优化器会根据当前参数的梯度和学习率来更新参数的值，从而使模型在训练过程中逐渐优化。
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward() # 在使用定制的优化器时，我们需要先调用backward()方法来计算损失函数的梯度，然后再调用updater函数来更新参数。这里我们使用l.sum()来计算总损失，因为在定制优化器中，我们通常需要对整个批次的损失进行求和，以便在更新参数时使用。
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]


lr = 0.1
def updater(batch_size):
    # 使用梯度下降法
    sgd([W, b], lr, batch_size) # W和b都要更新，所以将它们放在一个列表中传递给sgd函数，同时传递它们的梯度和学习率。

# lr = 0.1
# from d2l import torch as d2l
# def updater(batch_size):
#     # 使用梯度下降法
#     return d2l.sgd([W, b], lr, batch_size) 

def sgd(params, lr, batch_size):
    """Minibatch stochastic gradient descent.

    Defined in :numref:`sec_utils`"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size # grad属性包含了参数的梯度信息，通过这个信息我们可以更新参数的值。具体来说，param.grad是一个张量，表示当前参数的梯度。我们将这个梯度乘以学习率lr，并除以批量大小batch_size，以得到每个样本的平均梯度。然后，我们将这个平均梯度从当前参数值中减去，以更新参数的值。
            param.grad.zero_() # 在更新参数后，我们需要将参数的梯度清零，以便在下一次迭代中重新计算梯度。这个操作是必要的，因为在PyTorch中，默认情况下，梯度会累积在参数的.grad属性中，如果不清零，下一次计算梯度时会将新的梯度与之前的梯度相加，导致错误的更新。


class Animator:  #@save65
    """在动画中绘制数据"""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        # 增量地绘制多条线
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # 使用lambda函数捕获参数
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # 向图表中添加多个数据点
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)



def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):  #@save
    """训练模型（定义见第3章）"""
    animator = Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics
    assert train_loss < 0.5, train_loss
    assert train_acc <= 1 and train_acc > 0.7, train_acc
    assert test_acc <= 1 and test_acc > 0.7, test_acc